# Code Capsule: One-hot vs Dense Vectors (WP02)

**CS224N 2025 — Lecture 1: Word Vectors**

This notebook demonstrates why one-hot word vectors cannot encode semantic similarity,
and how dense vectors solve this problem.

**Official anchor**: slides p19-p20 (one-hot orthogonality); slides p22-p23 (dense vectors + dot product similarity); notes §2.2 Eq.1-Eq.2

---

## Part 1: One-hot Vectors and the Orthogonality Problem

In a one-hot encoding, each word gets its own unique position in a vector of size = vocabulary.
All other positions are zero. This means **every pair of different words has dot product = 0**,
regardless of how semantically similar they might be.

In [1]:
import numpy as np

# Small vocabulary with known semantic relationships
vocab = ['king', 'queen', 'cat', 'dog', 'bank', 'river', 'car', 'banana']
vocab_size = len(vocab)

# Build one-hot vectors
one_hot = {}
for i, word in enumerate(vocab):
    vec = np.zeros(vocab_size)
    vec[i] = 1.0
    one_hot[word] = vec

print(f'Vocabulary ({vocab_size} words): {vocab}')
print(f'One-hot vector dimension = vocabulary size = {vocab_size}')
print(f"\nExample: one_hot['king'] = {one_hot['king']}")
print(f"Example: one_hot['queen'] = {one_hot['queen']}")

Vocabulary (8 words): ['king', 'queen', 'cat', 'dog', 'bank', 'river', 'car', 'banana']
One-hot vector dimension = vocabulary size = 8

Example: one_hot['king'] = [1. 0. 0. 0. 0. 0. 0. 0.]
Example: one_hot['queen'] = [0. 1. 0. 0. 0. 0. 0. 0.]


In [2]:
# Compute pairwise dot products
pairs = [
    ('king', 'queen'),    # semantically similar (royalty)
    ('cat', 'dog'),       # semantically similar (pets)
    ('bank', 'river'),    # different domains
    ('car', 'banana'),    # unrelated
    ('king', 'king'),     # self
]

print(f"{'Word A':<10} {'Word B':<10} {'Dot Product':>12}")
print('-' * 35)
for w1, w2 in pairs:
    dot = float(np.dot(one_hot[w1], one_hot[w2]))
    print(f'{w1:<10} {w2:<10} {dot:>12.1f}')

print("\n>>> ALL off-diagonal dot products = 0.0")
print(">>> One-hot cannot distinguish 'king-queen' (similar) from 'car-banana' (unrelated)")

Word A     Word B      Dot Product
-----------------------------------
king       queen               0.0
cat        dog                 0.0
bank       river               0.0
car        banana              0.0
king       king                1.0

>>> ALL off-diagonal dot products = 0.0
>>> One-hot cannot distinguish 'king-queen' (similar) from 'car-banana' (unrelated)


## Part 2: Dense Vectors Can Encode Similarity

Instead of one-hot, we can learn **dense vectors** where each dimension captures some
aspect of meaning. Similar words get similar vectors, so their dot product / cosine
similarity is high.

In [3]:
# Hand-crafted 6D dense vectors (toy example)
# Dimensions: [royalty, animate, finance, nature, mechanical, food]
dense = {
    'king':   np.array([ 0.9,  0.6,  0.1,  0.0, -0.1, -0.2]),
    'queen':  np.array([ 0.85, 0.55, 0.1,  0.0, -0.1, -0.15]),
    'cat':    np.array([-0.1,  0.8,  0.0,  0.3, -0.2,  0.1]),
    'dog':    np.array([-0.05, 0.75, 0.0,  0.25,-0.15, 0.1]),
    'bank':   np.array([ 0.1, -0.3,  0.9, -0.2,  0.2, -0.1]),
    'river':  np.array([-0.1,  0.1, -0.2,  0.85, 0.0,  0.0]),
    'car':    np.array([-0.2, -0.2,  0.1, -0.3,  0.9,  0.0]),
    'banana': np.array([ 0.0,  0.2, -0.1,  0.3, -0.1,  0.85]),
}

def cosine_sim(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

print(f"{'Word A':<10} {'Word B':<10} {'Cosine Sim':>12} {'Relationship':<20}")
print('-' * 55)
for w1, w2 in pairs:
    cos = cosine_sim(dense[w1], dense[w2])
    rel = {'king-queen': 'similar (royalty)', 'cat-dog': 'similar (pets)',
           'bank-river': 'different domains', 'car-banana': 'unrelated',
           'king-king': 'self'}[f'{w1}-{w2}']
    print(f'{w1:<10} {w2:<10} {cos:>12.4f} {rel:<20}')

print("\n>>> Similar words (king-queen, cat-dog) have cosine ~ 1.0")
print(">>> Unrelated words (car-banana, bank-river) have cosine < 0")

Word A     Word B       Cosine Sim Relationship        
-------------------------------------------------------
king       queen            0.9992 similar (royalty)   
cat        dog              0.9971 similar (pets)      
bank       river           -0.4409 different domains   
car        banana          -0.2475 unrelated           
king       king             1.0000 self                

>>> Similar words (king-queen, cat-dog) have cosine ~ 1.0
>>> Unrelated words (car-banana, bank-river) have cosine < 0


## Part 3: Side-by-side Comparison

The table below makes the contrast crystal clear:
- **One-hot**: every non-self pair = 0.0 (no similarity information at all)
- **Dense**: similar words >> unrelated words (meaning is encoded in the vector)

In [4]:
print(f"{'Pair':<20} {'One-hot Dot':>12} {'Dense Cosine':>14}")
print('-' * 48)
for w1, w2 in pairs:
    oh = float(np.dot(one_hot[w1], one_hot[w2]))
    dc = cosine_sim(dense[w1], dense[w2])
    print(f'{w1}-{w2:<15} {oh:>12.1f} {dc:>14.4f}')

print("\n>>> This is why CS224N moves from one-hot to dense representations (slides p22)")

Pair                  One-hot Dot   Dense Cosine
------------------------------------------------
king-queen                    0.0         0.9992
cat-dog                      0.0         0.9971
bank-river                    0.0        -0.4409
car-banana                   0.0        -0.2475
king-king                     1.0         1.0000

>>> This is why CS224N moves from one-hot to dense representations (slides p22)
